In [ ]:
# Required libraries installation
!pip install rdkit-pypi
!pip install mordred
!pip install torch torchvision torchaudio
!pip install scikit-optimize
!pip install optuna
!pip install xgboost
!pip install lightgbm
!pip install shap
!pip uninstall -y patsy statsmodels seaborn
!pip install patsy
!pip install statsmodels
!pip install seaborn --upgrade
!pip install bayesian-optimization
!pip rdkit.Chem

!pip install -q condacolab
import condacolab
condacolab.install()

!conda install -c conda-forge rdkit -y

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.gridspec import GridSpec
import os
from matplotlib.colors import Normalize
import networkx as nx
import matplotlib.font_manager as fm

# Set matplotlib parameters
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = [10, 8]
plt.rcParams['figure.dpi'] = 150

# Results directory
os.makedirs('publication_figures', exist_ok=True)

from rdkit import Chem
from rdkit.Chem import AllChem
# from rdkit.Chem import MACCSkeys

# Setup for matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Method 1: Use default unicode font
plt.rcParams['font.family'] = 'DejaVu Sans'  # Unicode supported font
plt.rcParams['axes.unicode_minus'] = False   # Prevent minus sign issues

!pip install --force-reinstall --no-deps numpy==1.22.4
!pip install --force-reinstall --no-deps pandas matplotlib scikit-learn scipy rdkit-pypi
!pip install mordred torch xgboost lightgbm

  Using cached rdkit_pypi-2022.9.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.9 kB)
Using cached rdkit_pypi-2022.9.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (29.4 MB)
  Using cached mordred-1.2.0-py3-none-any.whl
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached networkx-2.8.8-py3-none-any.whl.metadata (5.1 kB)
Using cached networkx-2.8.8-py3-none-any.whl (2.0 MB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.0
    Uninstalling numpy-2.3.0:
      Successfully uninstalled numpy-2.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.22.4
    Uninstalling numpy-1.22.4:
      Successfully uninstalled numpy-1.22.4


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import time
import os
import joblib
from collections import Counter
import gc
import warnings
import json
from datetime import datetime
warnings.filterwarnings('ignore')

# sklearn related
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression, RFECV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from scipy import stats
from scipy.optimize import minimize

# XGBoost and LightGBM
import xgboost as xgb

# PyTorch related
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

# RDKit related
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import Descriptors3D
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import rdmolops
from rdkit.Chem import AllChem, Mol, GetSSSR, Descriptors3D

# Additional imports
import optuna
from sklearn.model_selection import KFold, cross_val_score
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
from functools import partial

# Mordred descriptor setup
from mordred import Calculator, descriptors

# GPU setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# RDKit log disable
RDLogger.DisableLog('rdApp.*')

Using device: cpu


In [ ]:
#------------------------------------------------------------------------------
# 1. Memory optimization function
#------------------------------------------------------------------------------

def clear_memory():
    """Memory cleanup function"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

#------------------------------------------------------------------------------
# 2. Checkpoint system for long-running experiments
#------------------------------------------------------------------------------

def load_csv_with_encoding(csv_file):
    """
    Load CSV file with automatic encoding detection
    """
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252', 'utf-16']

    for encoding in encodings:
        try:
            # Try reading with different encoding
            df = pd.read_csv(csv_file, encoding=encoding)
            print(f"Successfully loaded CSV with {encoding} encoding")

            # Check if required columns exist
            required_columns = ['Smiles', 'pChEMBL Value']
            missing_columns = [col for col in required_columns if col not in df.columns]

            if missing_columns:
                # Try case-insensitive column matching
                df_columns_lower = {col.lower(): col for col in df.columns}
                for req_col in missing_columns:
                    if req_col.lower() in df_columns_lower:
                        df.rename(columns={df_columns_lower[req_col.lower()]: req_col}, inplace=True)
                    else:
                        print(f"Warning: Required column '{req_col}' not found in CSV")
                        print(f"Available columns: {list(df.columns)}")

            # Remove any BOM characters from column names
            df.columns = df.columns.str.replace('\ufeff', '')
            df.columns = df.columns.str.strip()

            return df

        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Error with {encoding}: {e}")
            continue

    # If all encodings fail, try with error handling
    try:
        df = pd.read_csv(csv_file, encoding='utf-8', errors='ignore')
        print("Loaded CSV with utf-8 encoding (ignoring errors)")
        return df
    except Exception as e:
        raise ValueError(f"Could not read CSV file {csv_file}. Error: {e}")

class CheckpointManager:
    """Manages checkpoints for long-running experiments"""
    def __init__(self, checkpoint_dir='checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.checkpoint_file = os.path.join(checkpoint_dir, 'ablation_checkpoint.pkl')
        self.progress_file = os.path.join(checkpoint_dir, 'ablation_progress.json')

    def save_checkpoint(self, state_dict):
        """Save checkpoint with current state"""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        state_dict['timestamp'] = timestamp

        # Save main checkpoint
        joblib.dump(state_dict, self.checkpoint_file)

        # Save progress summary
        progress = {
            'timestamp': timestamp,
            'completed_experiments': state_dict.get('completed_experiments', []),
            'current_experiment': state_dict.get('current_experiment', None),
            'total_experiments': state_dict.get('total_experiments', 0),
            'elapsed_time': state_dict.get('elapsed_time', 0)
        }

        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=4)

        print(f"Checkpoint saved at {timestamp}")

    def load_checkpoint(self):
        """Load checkpoint if exists"""
        if os.path.exists(self.checkpoint_file):
            print(f"Loading checkpoint from {self.checkpoint_file}")
            return joblib.load(self.checkpoint_file)
        return None

    def checkpoint_exists(self):
        """Check if checkpoint exists"""
        return os.path.exists(self.checkpoint_file)

In [ ]:
#------------------------------------------------------------------------------
# 3. 3D molecular structure generation function
#------------------------------------------------------------------------------

def generate_3d_conformation(mol, max_attempts=3, num_conformers=5):
    """Optimized 3D structure generation function"""
    if mol is None:
        return None

    # Add hydrogen atoms
    mol = Chem.AddHs(mol)

    for attempt in range(max_attempts):
        try:
            # Generate multiple conformers using ETKDG method
            params = AllChem.ETKDGv3()
            params.numThreads = 0
            params.randomSeed = 42
            params.useRandomCoords = True
            AllChem.EmbedMultipleConfs(mol, numConfs=num_conformers, params=params)

            # Check if structures were generated
            if mol.GetNumConformers() > 0:
                # Optimize structures
                results = []
                for conf_id in range(mol.GetNumConformers()):
                    try:
                        # Structure optimization using MMFF94s force field
                        res = AllChem.MMFFOptimizeMolecule(mol, confId=conf_id, maxIters=1000)
                        results.append((conf_id, res))
                    except:
                        continue

                # Check if at least one structure is optimized
                if any(res == 0 for _, res in results):
                    # Calculate and sort by energy
                    try:
                        energies = []
                        for conf_id, _ in results:
                            if conf_id < mol.GetNumConformers():
                                try:
                                    # MMFF94 energy calculation
                                    mp = AllChem.MMFFGetMoleculeProperties(mol)
                                    ff = AllChem.MMFFGetMoleculeForceField(mol, mp, confId=conf_id)
                                    energies.append((conf_id, ff.CalcEnergy()))
                                except:
                                    continue

                        # Sort by energy and select lowest energy structure
                        if energies:
                            energies.sort(key=lambda x: x[1])
                            lowest_conf_id = energies[0][0]
                            lowest_conf_mol = Chem.Mol(mol)
                            lowest_conf_mol.RemoveAllConformers()
                            lowest_conf_mol.AddConformer(mol.GetConformer(lowest_conf_id))
                            return lowest_conf_mol
                    except:
                        pass

                # If energy calculation fails, return first conformer
                if mol.GetNumConformers() > 0:
                    first_conf_mol = Chem.Mol(mol)
                    first_conf_mol.RemoveAllConformers()
                    first_conf_mol.AddConformer(mol.GetConformer(0))
                    return first_conf_mol

        except Exception as e:
            if attempt == max_attempts - 1:
                print(f"Warning: 3D structure generation failed: {e}")
                return None
            continue

    return None

In [ ]:
#------------------------------------------------------------------------------
# 4. Descriptor calculation functions
#------------------------------------------------------------------------------

def get_essential_2d_descriptors(mol):
    """Optimized function to calculate only essential 2D descriptors"""
    if mol is None:
        return [0] * 50

    result_descriptors = []
    try:
        # 1. Basic druglikeness properties (10)
        result_descriptors.extend([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.RingCount(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumAromaticRings(mol),
            Descriptors.NumAliphaticRings(mol),
            Descriptors.FractionCSP3(mol)
        ])

        # 2. Atom count related (10)
        atom_counts = Counter([atom.GetSymbol() for atom in mol.GetAtoms()])
        result_descriptors.extend([
            atom_counts.get('C', 0),
            atom_counts.get('N', 0),
            atom_counts.get('O', 0),
            atom_counts.get('S', 0),
            atom_counts.get('F', 0),
            atom_counts.get('Cl', 0),
            atom_counts.get('Br', 0),
            mol.GetNumAtoms(),
            Descriptors.HeavyAtomCount(mol),
            Descriptors.NumHeteroatoms(mol)
        ])

        # 3. Bond/ring related properties (10)
        result_descriptors.extend([
            rdMolDescriptors.CalcNumRings(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcNumAliphaticRings(mol),
            rdMolDescriptors.CalcNumSaturatedRings(mol),
            rdMolDescriptors.CalcNumAliphaticHeterocycles(mol),
            rdMolDescriptors.CalcNumAromaticHeterocycles(mol),
            rdMolDescriptors.CalcNumSaturatedHeterocycles(mol),
            rdMolDescriptors.CalcNumSpiroAtoms(mol),
            rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
            Descriptors.qed(mol)
        ])

        # 4. Complexity and connectivity indices (10)
        result_descriptors.extend([
            Descriptors.BalabanJ(mol),
            Descriptors.BertzCT(mol),
            Descriptors.HallKierAlpha(mol),
            Descriptors.Kappa1(mol),
            Descriptors.Kappa2(mol),
            Descriptors.Chi0(mol),
            Descriptors.Chi1(mol),
            Descriptors.Chi2n(mol),
            Descriptors.Chi3n(mol),
            Descriptors.Chi4n(mol)
        ])

        # 5. Specific ratio related properties (10)
        num_atoms = mol.GetNumAtoms()
        num_bonds = mol.GetNumBonds()

        # Atom/bond ratios
        if num_atoms > 0:
            result_descriptors.append(num_bonds / num_atoms)
            aromatic_atoms = sum(1 for atom in mol.GetAtoms() if atom.GetIsAromatic())
            result_descriptors.append(aromatic_atoms / num_atoms)
            result_descriptors.append((atom_counts.get('N', 0) + atom_counts.get('O', 0)) / num_atoms)
            result_descriptors.append(Descriptors.NumHeteroatoms(mol) / num_atoms)
            result_descriptors.append(Descriptors.FractionCSP3(mol))
        else:
            result_descriptors.extend([0, 0, 0, 0, 0])

        # Bond type ratios
        if num_bonds > 0:
            bond_types = {
                Chem.BondType.SINGLE: 0,
                Chem.BondType.DOUBLE: 0,
                Chem.BondType.TRIPLE: 0,
                Chem.BondType.AROMATIC: 0
            }

            for bond in mol.GetBonds():
                bond_type = bond.GetBondType()
                if bond_type in bond_types:
                    bond_types[bond_type] += 1

            result_descriptors.extend([
                bond_types[Chem.BondType.SINGLE] / num_bonds,
                bond_types[Chem.BondType.DOUBLE] / num_bonds,
                bond_types[Chem.BondType.TRIPLE] / num_bonds,
                bond_types[Chem.BondType.AROMATIC] / num_bonds,
                Descriptors.NumRotatableBonds(mol) / num_bonds
            ])
        else:
            result_descriptors.extend([0, 0, 0, 0, 0])

    except Exception as e:
        print(f"Error calculating 2D descriptors: {e}")
        return [0] * 50

    # Replace missing values with 0
    result_descriptors = [0 if x is None or np.isnan(x) or np.isinf(x) else x for x in result_descriptors]

    return result_descriptors[:50]

def generate_ecfp_fingerprint(mol, radius=3, nBits=1024, useFeatures=True):
    """Optimized ECFP fingerprint calculation function"""
    if mol is None:
        return np.zeros(nBits, dtype=np.int32)

    try:
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius,
            nBits=nBits,
            useFeatures=useFeatures
        )
        return np.array(fp)
    except Exception as e:
        print(f"Error generating ECFP: {e}")
        return np.zeros(nBits, dtype=np.int32)

def generate_optimized_mordred_descriptors():
    """Setup for core Mordred descriptors relevant to GSK3-beta"""
    calc = Calculator([
        descriptors.RingCount,
        descriptors.Aromatic,
        descriptors.AtomCount,
        descriptors.HydrogenBond,
        descriptors.TopologicalCharge,
        descriptors.EState,
        descriptors.Polarizability,
        descriptors.TopoPSA,
        descriptors.LogS,
        descriptors.RotatableBond,
        descriptors.BertzCT,
        descriptors.WienerIndex,
        descriptors.Autocorrelation,
        descriptors.Chi,
        descriptors.Constitutional,
        descriptors.KappaShapeIndex,
        descriptors.CarbonTypes,
        descriptors.SLogP,
        descriptors.Weight
    ])
    return calc

def get_optimized_3d_descriptors(mol_3d):
    """Calculate only 3D descriptors important for GSK3-beta inhibitors"""
    descriptors_3d = []

    if mol_3d is not None and mol_3d.GetNumConformers() > 0:
        try:
            # 1. Molecular shape descriptors
            descriptors_3d.extend([
                Descriptors3D.Asphericity(mol_3d),
                Descriptors3D.Eccentricity(mol_3d),
                Descriptors3D.InertialShapeFactor(mol_3d),
                Descriptors3D.NPR1(mol_3d),
                Descriptors3D.NPR2(mol_3d),
                Descriptors3D.RadiusOfGyration(mol_3d),
                Descriptors3D.SpherocityIndex(mol_3d)
            ])

            # 2. Molecular size and volume
            try:
                mol_area = AllChem.ComputeMolSurfaceArea(mol_3d)
                volume = Descriptors3D.GetMolecularVolume(mol_3d)
                sa_vol_ratio = mol_area / max(0.001, volume)

                descriptors_3d.extend([
                    mol_area,
                    volume,
                    sa_vol_ratio
                ])
            except:
                descriptors_3d.extend([0, 0, 0])

            # 3. Interatomic distance statistics
            conf = mol_3d.GetConformer()
            positions = [conf.GetAtomPosition(i) for i in range(mol_3d.GetNumAtoms())]

            if len(positions) > 1:
                distances = []
                for i in range(len(positions)):
                    for j in range(i+1, len(positions)):
                        distances.append(positions[i].Distance(positions[j]))

                descriptors_3d.extend([
                    np.max(distances),
                    np.min(distances),
                    np.mean(distances),
                    np.std(distances)
                ])
            else:
                descriptors_3d.extend([0, 0, 0, 0])

            # Additional 3D properties (simplified for consistency)
            descriptors_3d.extend([0, 0, 0, 0, 0, 0])

        except Exception as e:
            print(f"Error calculating 3D descriptors: {e}")
            descriptors_3d = [0] * 20
    else:
        descriptors_3d = [0] * 20

    # Fix length to 20
    if len(descriptors_3d) < 20:
        descriptors_3d.extend([0] * (20 - len(descriptors_3d)))

    return descriptors_3d[:20]

def get_gsk3beta_specific_descriptors(mol):
    """Target-specific descriptors specialized for GSK3-beta inhibitors"""
    if mol is None:
        return [0] * 30

    specific_descriptors = []
    try:
        # 1. Hydrogen bonding properties
        hbond_donors = Lipinski.NumHDonors(mol)
        hbond_acceptors = Lipinski.NumHAcceptors(mol)
        heavy_atoms = Descriptors.HeavyAtomCount(mol)

        atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
        n_count = atom_symbols.count('N')
        o_count = atom_symbols.count('O')
        s_count = atom_symbols.count('S')

        heavy_atoms_safe = max(1, heavy_atoms)

        specific_descriptors.extend([
            hbond_donors,
            hbond_acceptors,
            hbond_donors / heavy_atoms_safe,
            hbond_acceptors / heavy_atoms_safe,
            (hbond_donors + hbond_acceptors) / heavy_atoms_safe
        ])

        # 2. Aromatic properties
        aromatic_rings = Descriptors.NumAromaticRings(mol)
        total_rings = rdMolDescriptors.CalcNumRings(mol)
        total_rings_safe = max(1, total_rings)

        specific_descriptors.extend([
            aromatic_rings,
            aromatic_rings / total_rings_safe,
            rdMolDescriptors.CalcNumAromaticCarbocycles(mol),
            rdMolDescriptors.CalcNumAromaticHeterocycles(mol)
        ])

        # 3. Atom type properties
        specific_descriptors.extend([
            n_count,
            o_count,
            s_count,
            n_count / heavy_atoms_safe,
            o_count / heavy_atoms_safe,
            s_count / heavy_atoms_safe
        ])

        # 4. Size and flexibility properties
        rotatable_bonds = Descriptors.NumRotatableBonds(mol)
        num_bonds = mol.GetNumBonds()
        num_bonds_safe = max(1, num_bonds)

        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)

        specific_descriptors.extend([
            rotatable_bonds,
            rotatable_bonds / num_bonds_safe,
            mw,
            logp,
            Descriptors.TPSA(mol)
        ])

        # 5. GSK3-beta specific composite indices
        hinge_score = (aromatic_rings * 0.5 +
                      hbond_donors * 0.3 +
                      hbond_acceptors * 0.2)

        hydrophobic_score = (logp * 0.6 +
                            aromatic_rings * 0.4)

        size_match = 1.0 - min(abs(mw - 400) / 400, 1.0)

        flexibility_penalty = min(rotatable_bonds / 10.0, 1.0)

        selectivity_score = (n_count * 0.3 +
                           aromatic_rings * 0.4 +
                           (hbond_donors + hbond_acceptors) * 0.3) / 10.0

        specific_descriptors.extend([
            hinge_score,
            hydrophobic_score,
            size_match,
            1.0 - flexibility_penalty,
            selectivity_score
        ])

        # 6. Charge-related properties
        formal_charges = [atom.GetFormalCharge() for atom in mol.GetAtoms()]
        pos_charges = sum(1 for c in formal_charges if c > 0)
        neg_charges = sum(1 for c in formal_charges if c < 0)

        n_atoms = [atom for atom in mol.GetAtoms() if atom.GetSymbol() == 'N']
        cationic_n = sum(1 for atom in n_atoms if atom.GetTotalNumHs() > 0)

        specific_descriptors.extend([
            pos_charges,
            neg_charges,
            cationic_n,
            pos_charges / heavy_atoms_safe,
            n_count * hbond_donors / (heavy_atoms_safe * 10)
        ])

    except Exception as e:
        print(f"Error calculating GSK3-beta specific descriptors: {e}")
        return [0] * 30

    specific_descriptors = [0 if x is None or np.isnan(x) or np.isinf(x) else x for x in specific_descriptors]

    return specific_descriptors[:30]

def enhanced_gsk3_descriptors(mol):
    """Enhanced GSK3-beta specific descriptors"""
    base_descriptors = get_gsk3beta_specific_descriptors(mol)

    atp_binding_descriptors = []
    try:
        hinge_donors = sum(1 for atom in mol.GetAtoms()
                       if atom.GetSymbol() in ['N', 'O'] and atom.GetTotalNumHs() > 0)
        hinge_acceptors = sum(1 for atom in mol.GetAtoms()
                         if atom.GetSymbol() in ['N', 'O'])

        logp = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        aromatic_ratio = Descriptors.NumAromaticRings(mol) / max(1, mol.GetNumAtoms())

        basic_n = sum(1 for atom in mol.GetAtoms()
                 if atom.GetSymbol() == 'N' and atom.GetTotalNumHs() > 0)
        basic_n_ratio = basic_n / max(1, mol.GetNumHeavyAtoms())

        cdk_selectivity = logp * tpsa / 1000.0

        atp_binding_descriptors.extend([
            hinge_donors, hinge_acceptors,
            hinge_donors / max(1, mol.GetNumHeavyAtoms()),
            hinge_acceptors / max(1, mol.GetNumHeavyAtoms()),
            logp / max(1, tpsa),
            aromatic_ratio,
            basic_n, basic_n_ratio,
            cdk_selectivity
        ])
    except:
        atp_binding_descriptors = [0] * 9

    dfg_descriptors = []
    try:
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        rings = rdMolDescriptors.CalcNumRings(mol)
        flexibility = rot_bonds / max(1, mol.GetNumBonds())

        arom_rings = Descriptors.NumAromaticRings(mol)
        arom_het_rings = rdMolDescriptors.CalcNumAromaticHeterocycles(mol)

        dfg_descriptors.extend([
            flexibility,
            1.0 - flexibility,
            arom_rings,
            arom_het_rings,
            arom_rings / max(1, rings) if rings > 0 else 0
        ])
    except:
        dfg_descriptors = [0] * 5

    return np.concatenate([base_descriptors, atp_binding_descriptors, dfg_descriptors])

In [ ]:
#------------------------------------------------------------------------------
# 5. Improved neural network model
#------------------------------------------------------------------------------

class ImprovedGSK3NeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128, 64, 32], dropout_rates=[0.3, 0.3, 0.2, 0.2]):
        super(ImprovedGSK3NeuralNetwork, self).__init__()

        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.dropout_rates = dropout_rates

        self.input_bn = nn.BatchNorm1d(input_dim)

        self.layers = nn.ModuleList()
        prev_dim = input_dim

        for i, (hdim, dropout) in enumerate(zip(hidden_dims, dropout_rates)):
            linear = nn.Linear(prev_dim, hdim)
            self.layers.append(linear)
            self.layers.append(nn.BatchNorm1d(hdim))
            self.layers.append(nn.SiLU())
            self.layers.append(nn.Dropout(dropout))
            prev_dim = hdim

        self.output = nn.Linear(hidden_dims[-1], 1)
        self._init_weights()

    def _init_weights(self):
        """Weight initialization"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_bn(x)
        for layer in self.layers:
            x = layer(x)
        return self.output(x)


In [ ]:
#------------------------------------------------------------------------------
# 6. Robust Feature Pipeline
#------------------------------------------------------------------------------

class RobustFeaturePipeline:
    def __init__(self):
        self.scaler = None
        self.non_constant_mask = None
        self.feature_selector = None
        self.is_fitted = False
        self.original_feature_dim = None
        self.selected_indices_ = None
        self.mordred_columns = None

    def fit(self, X, y=None):
        self.original_feature_dim = X.shape[1]
        self.non_constant_mask = np.std(X, axis=0) != 0
        X_filtered = X[:, self.non_constant_mask]
        self.scaler = RobustScaler()
        X_scaled = self.scaler.fit_transform(X_filtered)

        if y is not None and self.feature_selector:
            X_selected = self.feature_selector.fit_transform(X_scaled, y)
        else:
            X_selected = X_scaled

        self.is_fitted = True
        return X_selected

    def transform(self, X):
        if not self.is_fitted:
            raise ValueError("Pipeline not fitted. Call fit method first.")

        if X.shape[1] != self.original_feature_dim:
            print(f"Input dimension mismatch: current {X.shape[1]}, expected {self.original_feature_dim}")

            if X.shape[1] > self.original_feature_dim:
                print(f"Reducing input dimension: {X.shape[1]} -> {self.original_feature_dim}")
                X = X[:, :self.original_feature_dim]
            else:
                print(f"Expanding input dimension: {X.shape[1]} -> {self.original_feature_dim}")
                padding = np.zeros((X.shape[0], self.original_feature_dim - X.shape[1]))
                X = np.hstack([X, padding])

            print(f"Adjusted input dimension: {X.shape[1]}")

        if self.non_constant_mask is not None:
            if len(self.non_constant_mask) != X.shape[1]:
                print(f"Mask dimension mismatch: mask {len(self.non_constant_mask)}, input {X.shape[1]}")
                self.non_constant_mask = np.ones(X.shape[1], dtype=bool)
                print("Mask reset: keeping all features")
        else:
            self.non_constant_mask = np.ones(X.shape[1], dtype=bool)

        X_filtered = X[:, self.non_constant_mask]
        print(f"After removing constant features: {X_filtered.shape[1]}")

        if self.scaler is not None:
            try:
                X_scaled = self.scaler.transform(X_filtered)
            except Exception as e:
                print(f"Scaling error: {e}, using default scaler")
                self.scaler = StandardScaler()
                X_scaled = self.scaler.fit_transform(X_filtered)
        else:
            X_scaled = X_filtered

        if hasattr(self, 'selected_indices_') and self.selected_indices_ is not None:
            max_idx = np.max(self.selected_indices_)

            if max_idx < X_scaled.shape[1]:
                X_selected = X_scaled[:, self.selected_indices_]
                print(f"After feature selection: {X_selected.shape[1]}")
            else:
                print(f"Feature index out of range (max index: {max_idx}, features: {X_scaled.shape[1]})")
                valid_indices = self.selected_indices_[self.selected_indices_ < X_scaled.shape[1]]

                if len(valid_indices) > 0:
                    X_selected = X_scaled[:, valid_indices]
                    print(f"Using valid indices only: {X_selected.shape[1]}")
                else:
                    print("No valid indices, using all features")
                    X_selected = X_scaled
        else:
            X_selected = X_scaled

        return X_selected

    def fit_transform(self, X, y=None):
        return self.fit(X, y)

    def save(self, filepath):
        pipeline_data = {
            'original_feature_dim': self.original_feature_dim,
            'scaler': self.scaler,
            'non_constant_mask': self.non_constant_mask,
            'feature_selector': self.feature_selector,
            'is_fitted': self.is_fitted,
            'selected_indices_': getattr(self, 'selected_indices_', None),
            'mordred_columns': getattr(self, 'mordred_columns', None)
        }
        joblib.dump(pipeline_data, filepath)
        print(f"Feature pipeline saved to {filepath}")

    @classmethod
    def load(cls, filepath):
        pipeline_data = joblib.load(filepath)
        pipeline = cls()
        pipeline.original_feature_dim = pipeline_data.get('original_feature_dim')
        pipeline.scaler = pipeline_data.get('scaler')
        pipeline.non_constant_mask = pipeline_data.get('non_constant_mask')
        pipeline.feature_selector = pipeline_data.get('feature_selector')
        pipeline.is_fitted = pipeline_data.get('is_fitted', False)

        if 'selected_indices_' in pipeline_data:
            pipeline.selected_indices_ = pipeline_data['selected_indices_']
            print(f"Feature pipeline loaded from {filepath} (selected features: {len(pipeline.selected_indices_)})")
        else:
            print(f"Feature pipeline loaded from {filepath} (no selected features info)")

        if 'mordred_columns' in pipeline_data:
            pipeline.mordred_columns = pipeline_data['mordred_columns']

        return pipeline

In [ ]:
#------------------------------------------------------------------------------
# 7. Adaptive Weight Ensemble
#------------------------------------------------------------------------------

class AdaptiveWeightEnsemble:
    """Adaptive ensemble with performance-based weights"""
    def __init__(self, base_models=None, meta_model=None):
        self.base_models = base_models or [
            RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_split=10, min_samples_leaf=5, random_state=42),
            xgb.XGBRegressor(n_estimators=200, learning_rate=0.02, max_depth=6, subsample=0.7, colsample_bytree=0.7, random_state=42),
            SVR(kernel='rbf', C=5.0, gamma='scale', epsilon=0.1),
            KNeighborsRegressor(n_neighbors=7, weights='distance'),
            ElasticNet(alpha=0.05, l1_ratio=0.7, max_iter=10000)
        ]

        self.meta_model = meta_model
        self.model_weights = None
        self.nn_model = None
        self.nn_weight = 0.0
        self.is_fitted = False
        self.y_scaler = None

    def fit(self, X_train, y_train, X_val, y_val):
        """Train ensemble with adaptive weights"""
        y_mean, y_std = np.mean(y_train), np.std(y_train)
        self.y_scaler = {'mean': y_mean, 'std': y_std}
        y_train_scaled = (y_train - y_mean) / y_std

        n_models = len(self.base_models)
        model_preds = np.zeros((X_val.shape[0], n_models))
        model_scores = []

        for i, model in enumerate(self.base_models):
            try:
                if hasattr(model, 'fit') and 'sample_weight' in model.fit.__code__.co_varnames:
                    weights = 1.0 + 0.2 * np.abs(y_train)
                    model.fit(X_train, y_train, sample_weight=weights)
                else:
                    model.fit(X_train, y_train)

                preds = model.predict(X_val)
                model_preds[:, i] = preds

                score = r2_score(y_val, preds)
                model_scores.append(max(0, score))

                print(f"  Model {i+1} ({model.__class__.__name__}) - Validation R2: {score:.4f}")
            except Exception as e:
                print(f"  Model {i+1} ({model.__class__.__name__}) - Error: {e}")
                model_scores.append(0)
                model_preds[:, i] = np.zeros(X_val.shape[0])

        try:
            if self.meta_model is None:
                self.meta_model = GradientBoostingRegressor(n_estimators=80, learning_rate=0.04,
                                                        max_depth=2, subsample=0.7, random_state=42)

            self.meta_model.fit(model_preds, y_val)
            meta_preds = self.meta_model.predict(model_preds)
            meta_score = r2_score(y_val, meta_preds)
            print(f"  Meta model - Validation R2: {meta_score:.4f}")
        except Exception as e:
            print(f"  Meta model training error: {e}")

        if self.nn_model is not None:
            self.nn_model.eval()
            try:
                with torch.no_grad():
                    X_val_tensor = torch.FloatTensor(X_val).to(device)
                    nn_preds = self.nn_model(X_val_tensor).cpu().numpy().flatten()

                nn_score = r2_score(y_val, nn_preds)
                print(f"  Neural network model - Validation R2: {nn_score:.4f}")

                if nn_score > 0:
                    if nn_score > np.max(model_scores) + 0.05:
                        self.nn_weight = 0.7
                    elif nn_score > np.max(model_scores):
                        self.nn_weight = 0.6
                    elif nn_score > np.mean(model_scores) + 0.05:
                        self.nn_weight = 0.5
                    elif nn_score > np.mean(model_scores):
                        self.nn_weight = 0.4
                    else:
                        self.nn_weight = 0.3
                else:
                    self.nn_weight = 0.0

                print(f"  Neural network weight: {self.nn_weight:.2f}")
            except Exception as e:
                print(f"  Neural network prediction error: {e}")
                self.nn_weight = 0.0

        try:
            self.model_weights = self.optimal_ensemble_weights(model_preds, y_val, method='bayesian')
            print(f"  Optimized model weights: {[f'{w:.4f}' for w in self.model_weights]}")
        except Exception as e:
            print(f"  Weight optimization error ({e}), using performance-based weights")

            total_score = sum(model_scores)
            if total_score > 0:
                self.model_weights = [score/total_score for score in model_scores]
            else:
                self.model_weights = [1.0/n_models] * n_models
                print("  Using equal weights (all models underperformed)")

        self.is_fitted = True
        return self

    def optimal_ensemble_weights(self, model_preds, y_true, method='bayesian'):
        """Find optimal ensemble weights using Bayesian optimization"""
        if method == 'bayesian':
            def objective(weights):
                weights = np.array(weights)
                weights = weights / np.sum(weights)
                ensemble_pred = np.sum(model_preds * weights, axis=1)
                return -r2_score(y_true, ensemble_pred)

            constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
            bounds = [(0, 1) for _ in range(model_preds.shape[1])]
            initial_weights = np.ones(model_preds.shape[1]) / model_preds.shape[1]

            result = minimize(
                objective,
                initial_weights,
                method='SLSQP',
                bounds=bounds,
                constraints=constraints
            )

            optimal_weights = result.x
            optimal_weights = optimal_weights / np.sum(optimal_weights)

            return optimal_weights
        else:
            scores = []
            for i in range(model_preds.shape[1]):
                score = r2_score(y_true, model_preds[:, i])
                scores.append(max(0, score))

            total_score = sum(scores)
            if total_score > 0:
                weights = [score/total_score for score in scores]
            else:
                weights = [1.0/model_preds.shape[1]] * model_preds.shape[1]

            return weights

    def predict(self, X):
        """Make ensemble predictions"""
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")

        n_models = len(self.base_models)
        n_samples = X.shape[0]
        model_preds = np.zeros((n_samples, n_models))

        for i, model in enumerate(self.base_models):
            try:
                model_preds[:, i] = model.predict(X)

                if np.any(np.isnan(model_preds[:, i])) or np.any(np.isinf(model_preds[:, i])):
                    print(f"Model {i} predictions contain NaN/Inf values. Replacing with mean")
                    model_preds[:, i] = np.nan_to_num(model_preds[:, i], nan=np.nanmean(model_preds[:, i]))
            except Exception as e:
                print(f"Model {i} prediction error: {e}")
                valid_preds = [model_preds[:, j] for j in range(i) if not np.any(np.isnan(model_preds[:, j]))]
                if valid_preds:
                    model_preds[:, i] = np.mean(valid_preds, axis=0)
                else:
                    model_preds[:, i] = self.y_scaler['mean'] if self.y_scaler else 0

        try:
            stacking_pred = self.meta_model.predict(model_preds)
        except Exception as e:
            print(f"Meta model prediction error: {e}. Using weighted average")
            stacking_pred = np.sum(model_preds * np.array(self.model_weights), axis=1)

        if self.nn_model is not None and self.nn_weight > 0:
            try:
                self.nn_model.eval()
                with torch.no_grad():
                    X_tensor = torch.FloatTensor(X).to(device)
                    nn_pred = self.nn_model(X_tensor).cpu().numpy().flatten()

                if np.any(np.isnan(nn_pred)) or np.any(np.isinf(nn_pred)):
                    print("Neural network predictions contain NaN/Inf values. Applying correction")
                    nn_pred = np.nan_to_num(nn_pred, nan=self.y_scaler['mean'] if self.y_scaler else 0)

                final_pred = (1 - self.nn_weight) * stacking_pred + self.nn_weight * nn_pred
            except Exception as e:
                print(f"Neural network prediction integration error: {e}")
                final_pred = stacking_pred
        else:
            final_pred = stacking_pred

        final_pred = np.clip(final_pred, 4.0, 10.0)

        return final_pred

In [ ]:
#------------------------------------------------------------------------------
# 8. mRMR-based feature selection
#------------------------------------------------------------------------------

def objective_group_importance_selection(X_train_scaled, y_train, new_feature_groups):
    """
    mRMR (Minimum Redundancy Maximum Relevance) algorithm for optimal feature selection
    """
    print("\nPerforming objective mRMR-based feature importance evaluation...")

    from sklearn.feature_selection import mutual_info_regression

    n_features = X_train_scaled.shape[1]
    min_features = 200

    print("  Calculating feature-target mutual information...")
    relevance = mutual_info_regression(X_train_scaled, y_train, random_state=42)
    relevance = np.nan_to_num(relevance)

    min_group_features = {
        'ECFP': max(50, int(0.1 * new_feature_groups.get('ECFP', 0))),
        '2D': max(10, int(0.2 * new_feature_groups.get('2D', 0))),
        'GSK3': max(15, int(0.3 * new_feature_groups.get('GSK3', 0))),
        '3D': max(10, int(0.5 * new_feature_groups.get('3D', 0))),
        'Mordred': max(30, int(0.1 * new_feature_groups.get('Mordred', 0)))
    }

    selected_features = []
    start_idx = 0

    print("  Selecting minimum features per group...")
    for group, count in new_feature_groups.items():
        end_idx = start_idx + count
        group_relevance = relevance[start_idx:end_idx]

        n_select = min(min_group_features.get(group, 5), count)

        if n_select > 0 and len(group_relevance) > 0:
            top_indices = np.argsort(group_relevance)[-n_select:]
            group_selected = [idx + start_idx for idx in top_indices]
            selected_features.extend(group_selected)
            print(f"  Group {group}: {n_select} features initially selected")

        start_idx = end_idx

    remaining = min_features - len(selected_features)

    if remaining > 0:
        print(f"  Selecting additional {remaining} features using mRMR...")

        candidates = np.array([i for i in range(n_features) if i not in selected_features])

        for i in range(remaining):
            if len(candidates) == 0:
                break

            mrmr_scores = np.zeros(len(candidates))

            for j, candidate in enumerate(candidates):
                relevance_term = relevance[candidate]

                if len(selected_features) > 0:
                    redundancy_values = []

                    sample_selected = selected_features
                    if len(selected_features) > 100:
                        sample_selected = np.random.choice(selected_features, 100, replace=False)

                    x_candidate = X_train_scaled[:, candidate].reshape(-1, 1)

                    for selected in sample_selected:
                        x_selected = X_train_scaled[:, selected].reshape(-1, 1)
                        mi = mutual_info_regression(x_candidate, X_train_scaled[:, selected])[0]
                        redundancy_values.append(mi)

                    redundancy = np.mean(redundancy_values) if redundancy_values else 0
                else:
                    redundancy = 0

                mrmr_scores[j] = relevance_term - redundancy

            if len(mrmr_scores) > 0:
                best_idx = np.argmax(mrmr_scores)
                selected_features.append(candidates[best_idx])
                candidates = np.delete(candidates, best_idx)

                if (i+1) % (remaining//10 or 1) == 0:
                    print(f"  mRMR progress: {i+1}/{remaining} features selected")

    selected_features = np.array(selected_features)

    feature_distribution = {group: 0 for group in new_feature_groups}
    start_idx = 0
    for group, count in new_feature_groups.items():
        end_idx = start_idx + count
        group_range = range(start_idx, end_idx)

        feature_distribution[group] = sum(1 for idx in selected_features if idx in group_range)
        start_idx = end_idx

    print("\nFeature distribution selected by mRMR:")
    for group, count in feature_distribution.items():
        ratio = count / len(selected_features) * 100
        print(f"  {group}: {count} features ({ratio:.1f}%)")

    importance_threshold = 0

    return selected_features, feature_distribution, importance_threshold

In [ ]:
#------------------------------------------------------------------------------
# 9. Ablation Study with Checkpoint Support
#------------------------------------------------------------------------------

def run_ablation_study_with_checkpoints(X_combined, y, feature_groups, output_dir='gsk3_results',
                                       n_cv_folds=5, checkpoint_interval=3600):
    """
    Run ablation study with checkpoint support for long-running experiments

    Args:
        X_combined: Combined feature matrix
        y: Target values
        feature_groups: Dictionary of feature group names and sizes
        output_dir: Output directory
        n_cv_folds: Number of cross-validation folds
        checkpoint_interval: Time interval (seconds) between checkpoints
    """

    # Create directories
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"{output_dir}/ablation", exist_ok=True)
    os.makedirs(f"{output_dir}/models", exist_ok=True)

    # Initialize checkpoint manager
    checkpoint_manager = CheckpointManager(f"{output_dir}/checkpoints")

    # Load checkpoint if exists
    checkpoint = checkpoint_manager.load_checkpoint()

    if checkpoint is not None:
        print("Resuming from checkpoint...")
        ablation_results = checkpoint['ablation_results']
        completed_experiments = checkpoint['completed_experiments']
        start_time = time.time() - checkpoint['elapsed_time']
        print(f"Completed experiments: {completed_experiments}")
    else:
        print("Starting new ablation study...")
        ablation_results = {
            'experiment': [],
            'r2_mean': [],
            'r2_std': [],
            'rmse_mean': [],
            'rmse_std': [],
            'removed_features': [],
            'training_time': []
        }
        completed_experiments = []
        start_time = time.time()

    # Calculate feature group indices
    group_indices = {}
    start_idx = 0
    for group, count in feature_groups.items():
        end_idx = start_idx + count
        group_indices[group] = list(range(start_idx, end_idx))
        start_idx = end_idx

    # Setup experiments
    experiments = {'All Features': None}
    for group in feature_groups.keys():
        experiments[f'No {group}'] = group

    total_experiments = len(experiments)
    last_checkpoint_time = time.time()

    # Run experiments
    for exp_name, excluded_group in experiments.items():
        # Skip if already completed
        if exp_name in completed_experiments:
            print(f"\nExperiment '{exp_name}' already completed, skipping...")
            continue

        print(f"\n{'='*60}")
        print(f"Running experiment: {exp_name}")
        print(f"Progress: {len(completed_experiments)}/{total_experiments} experiments completed")
        print(f"{'='*60}")

        exp_start_time = time.time()

        # Feature selection
        if excluded_group is not None:
            excluded_indices = group_indices[excluded_group]
            included_indices = [i for i in range(X_combined.shape[1]) if i not in excluded_indices]
            X_selected = X_combined[:, included_indices]
            removed_features = len(excluded_indices)
            print(f"Excluded feature group: {excluded_group} ({removed_features} features)")
        else:
            X_selected = X_combined
            removed_features = 0

        print(f"Selected features: {X_selected.shape[1]}")

        # Cross-validation
        kf = KFold(n_splits=n_cv_folds, shuffle=True, random_state=42)
        r2_scores = []
        rmse_scores = []

        for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected)):
            fold_start_time = time.time()

            X_train, X_test = X_selected[train_idx], X_selected[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Apply preprocessing pipeline
            feature_pipeline = RobustFeaturePipeline()
            feature_pipeline.original_feature_dim = X_train.shape[1]

            # Remove constant features
            feature_pipeline.non_constant_mask = np.std(X_train, axis=0) != 0
            X_train_filtered = X_train[:, feature_pipeline.non_constant_mask]
            X_test_filtered = X_test[:, feature_pipeline.non_constant_mask]

            # Scaling
            feature_pipeline.scaler = RobustScaler()
            X_train_scaled = feature_pipeline.scaler.fit_transform(X_train_filtered)
            X_test_scaled = feature_pipeline.scaler.transform(X_test_filtered)

            # Train model (using XGBoost for efficiency)
            model = xgb.XGBRegressor(
                n_estimators=200,
                learning_rate=0.02,
                max_depth=5,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_alpha=0.05,
                reg_lambda=1.0,
                random_state=42
            )

            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)

            # Evaluate
            r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))

            r2_scores.append(r2)
            rmse_scores.append(rmse)

            fold_time = time.time() - fold_start_time
            print(f"  Fold {fold+1}: R2 = {r2:.4f}, RMSE = {rmse:.4f} (Time: {fold_time:.2f}s)")

        # Calculate average performance
        r2_mean = np.mean(r2_scores)
        r2_std = np.std(r2_scores)
        rmse_mean = np.mean(rmse_scores)
        rmse_std = np.std(rmse_scores)

        exp_time = time.time() - exp_start_time

        print(f"  Average: R2 = {r2_mean:.4f} ± {r2_std:.4f}, RMSE = {rmse_mean:.4f} ± {rmse_std:.4f}")
        print(f"  Experiment time: {exp_time:.2f} seconds ({exp_time/60:.2f} minutes)")

        # Save results
        ablation_results['experiment'].append(exp_name)
        ablation_results['r2_mean'].append(r2_mean)
        ablation_results['r2_std'].append(r2_std)
        ablation_results['rmse_mean'].append(rmse_mean)
        ablation_results['rmse_std'].append(rmse_std)
        ablation_results['removed_features'].append(removed_features)
        ablation_results['training_time'].append(exp_time)

        completed_experiments.append(exp_name)

        # Check if checkpoint is needed
        current_time = time.time()
        if current_time - last_checkpoint_time > checkpoint_interval:
            elapsed_time = current_time - start_time

            checkpoint_state = {
                'ablation_results': ablation_results,
                'completed_experiments': completed_experiments,
                'current_experiment': exp_name,
                'total_experiments': total_experiments,
                'elapsed_time': elapsed_time,
                'feature_groups': feature_groups,
                'X_shape': X_combined.shape,
                'y_shape': y.shape
            }

            checkpoint_manager.save_checkpoint(checkpoint_state)
            last_checkpoint_time = current_time

            print(f"\nCheckpoint saved. Total elapsed time: {elapsed_time:.2f}s ({elapsed_time/3600:.2f}h)")

    # Final save
    total_time = time.time() - start_time

    # Save final results
    results_df = pd.DataFrame(ablation_results)
    results_df.to_csv(f"{output_dir}/ablation/ablation_results.csv", index=False)

    # Create visualizations
    plot_ablation_results(ablation_results, f"{output_dir}/ablation")

    # Save summary
    summary = {
        'total_experiments': total_experiments,
        'total_time_seconds': total_time,
        'total_time_hours': total_time / 3600,
        'average_time_per_experiment': total_time / total_experiments,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    with open(f"{output_dir}/ablation/ablation_summary.json", 'w') as f:
        json.dump(summary, f, indent=4)

    print(f"\nAblation study completed!")
    print(f"Total time: {total_time:.2f} seconds ({total_time/3600:.2f} hours)")
    print(f"Results saved to {output_dir}/ablation/")

    return ablation_results

def plot_ablation_results(ablation_results, output_dir):
    """Create visualizations for ablation study results"""

    results_df = pd.DataFrame(ablation_results)

    # Sort results
    base_result = results_df[results_df['experiment'] == 'All Features']
    other_results = results_df[results_df['experiment'] != 'All Features'].sort_values('r2_mean', ascending=False)
    results_df = pd.concat([base_result, other_results]).reset_index(drop=True)

    # 1. R2 performance plot
    plt.figure(figsize=(12, 6))
    plt.errorbar(
        results_df['experiment'],
        results_df['r2_mean'],
        yerr=results_df['r2_std'],
        fmt='o',
        capsize=5,
        ecolor='red',
        markersize=8
    )

    base_r2 = results_df.loc[0, 'r2_mean']
    plt.axhline(y=base_r2, color='gray', linestyle='--', alpha=0.7)

    plt.title('R2 Performance by Feature Group Exclusion', fontsize=15)
    plt.ylabel('R2 Score (mean ± std)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.xticks(rotation=45)

    plt.savefig(f"{output_dir}/ablation_r2.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 2. RMSE performance plot
    plt.figure(figsize=(12, 6))
    plt.errorbar(
        results_df['experiment'],
        results_df['rmse_mean'],
        yerr=results_df['rmse_std'],
        fmt='o',
        capsize=5,
        ecolor='red',
        markersize=8
    )

    base_rmse = results_df.loc[0, 'rmse_mean']
    plt.axhline(y=base_rmse, color='gray', linestyle='--', alpha=0.7)

    plt.title('RMSE Performance by Feature Group Exclusion', fontsize=15)
    plt.ylabel('RMSE (mean ± std)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.xticks(rotation=45)

    plt.savefig(f"{output_dir}/ablation_rmse.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 3. Feature importance ranking
    plt.figure(figsize=(10, 6))

    r2_decrease = []
    labels = []

    for i, row in results_df[results_df['experiment'] != 'All Features'].iterrows():
        decrease = base_r2 - row['r2_mean']
        r2_decrease.append(decrease)
        group_name = row['experiment'].replace('No ', '')
        labels.append(group_name)

    sorted_indices = np.argsort(r2_decrease)[::-1]
    r2_decrease = [r2_decrease[i] for i in sorted_indices]
    labels = [labels[i] for i in sorted_indices]

    plt.barh(labels, r2_decrease, color='skyblue')
    plt.xlabel('R2 Decrease (when feature group is removed)', fontsize=12)
    plt.title('Feature Group Importance', fontsize=15)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    for i, v in enumerate(r2_decrease):
        plt.text(v + 0.001, i, f'{v:.4f}', va='center')

    plt.savefig(f"{output_dir}/feature_group_importance.png", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Ablation study visualizations saved to {output_dir}")

In [ ]:
#------------------------------------------------------------------------------
# 10. Main pipeline function
#------------------------------------------------------------------------------

def run_mrmr_pipeline_with_ablation(csv_file, output_dir='gsk3_results', run_ablation=True):
    """
    Run the complete GSK3-beta QSAR pipeline with mRMR feature selection and optional ablation study
    """

    start_time = time.time()

    # Create directories
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"{output_dir}/models", exist_ok=True)
    os.makedirs(f"{output_dir}/visualizations", exist_ok=True)

    # 1. Load and preprocess data with encoding handling
    print(f"Loading data from {csv_file}...")

    try:
        df = load_csv_with_encoding(csv_file)
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        raise

    # Check for required columns
    if 'pChEMBL Value' not in df.columns:
        print("Available columns:", df.columns.tolist())
        raise ValueError("Required column 'pChEMBL Value' not found in CSV")

    if 'Smiles' not in df.columns:
        print("Available columns:", df.columns.tolist())
        raise ValueError("Required column 'Smiles' not found in CSV")

    df = df.dropna(subset=['pChEMBL Value'])
    print(f"Working with {len(df)} compounds after preprocessing")

    # 2. Generate molecular structures
    print("Generating molecular structures...")
    mols_2d = [Chem.MolFromSmiles(smi) for smi in df['Smiles']]

    print("Generating 3D structures...")
    mols_3d = [generate_3d_conformation(mol) for mol in mols_2d]
    successful_3d = sum(1 for mol in mols_3d if mol is not None and mol.GetNumConformers() > 0)
    print(f"3D structure generation success rate: {successful_3d}/{len(mols_2d)} ({successful_3d/len(mols_2d)*100:.2f}%)")

    # 3. Calculate descriptors
    print("Calculating descriptors...")

    print("Calculating fingerprints...")
    fps = [generate_ecfp_fingerprint(mol, radius=3, nBits=1024) for mol in mols_2d]
    fps = np.array(fps)
    print(f"Fingerprint size: {fps.shape}")

    print("Calculating 2D descriptors...")
    descriptors_2d = [get_essential_2d_descriptors(mol) for mol in mols_2d]
    descriptors_2d = np.array(descriptors_2d)
    print(f"2D descriptor size: {descriptors_2d.shape}")

    print("Calculating GSK3-specific descriptors...")
    gsk3_desc = [enhanced_gsk3_descriptors(mol) for mol in mols_2d]
    gsk3_desc = np.array(gsk3_desc)
    print(f"GSK3 descriptor size: {gsk3_desc.shape}")

    print("Calculating 3D descriptors...")
    descriptors_3d = [get_optimized_3d_descriptors(mol) for mol in mols_3d]
    descriptors_3d = np.array(descriptors_3d)
    print(f"3D descriptor size: {descriptors_3d.shape}")

    print("Calculating Mordred descriptors...")
    calc = generate_optimized_mordred_descriptors()
    mordred_df = calc.pandas(mols_2d, quiet=True).fillna(0)
    numeric_cols = mordred_df.select_dtypes(include=[np.number]).columns
    mordred_desc = mordred_df[numeric_cols].values
    mordred_columns = list(numeric_cols)
    print(f"Mordred descriptor size: {mordred_desc.shape}")

    # Feature group information
    feature_groups = {
        'ECFP': fps.shape[1],
        '2D': descriptors_2d.shape[1],
        'GSK3': gsk3_desc.shape[1],
        '3D': descriptors_3d.shape[1],
        'Mordred': mordred_desc.shape[1]
    }

    print("Feature group sizes:")
    for group, size in feature_groups.items():
        print(f"  {group}: {size}")

    # 4. Combine features
    print("\nCombining all features...")
    X_combined = np.hstack([
        fps,
        descriptors_2d,
        gsk3_desc,
        descriptors_3d,
        mordred_desc
    ])
    print(f"Combined feature matrix size: {X_combined.shape}")

    y = df['pChEMBL Value'].values

    # 5. Run ablation study if requested
    if run_ablation:
        print("\n" + "="*60)
        print("RUNNING ABLATION STUDY")
        print("="*60)

        ablation_results = run_ablation_study_with_checkpoints(
            X_combined, y, feature_groups, output_dir,
            n_cv_folds=5, checkpoint_interval=3600
        )

        # Print summary
        print("\nAblation Study Summary:")
        base_r2 = None
        for exp, r2 in zip(ablation_results['experiment'], ablation_results['r2_mean']):
            if exp == 'All Features':
                base_r2 = r2
                print(f"  Baseline (All Features): R2 = {r2:.4f}")
            else:
                r2_decrease = base_r2 - r2
                group = exp.replace('No ', '')
                print(f"  Without {group}: R2 = {r2:.4f} (decrease: {r2_decrease:.4f})")

    # 6. Continue with regular pipeline
    print("\n" + "="*60)
    print("RUNNING MAIN QSAR PIPELINE")
    print("="*60)

    # Data split
    print("\nSplitting data...")
    y_bins = pd.qcut(y, q=5, labels=False, duplicates='drop')
    X_train, X_test, y_train, y_test, y_bins_train, y_bins_test = train_test_split(
        X_combined, y, y_bins, test_size=0.2, random_state=42, stratify=y_bins
    )
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")

    # Feature preprocessing
    print("\nApplying feature preprocessing...")
    feature_pipeline = RobustFeaturePipeline()
    feature_pipeline.original_feature_dim = X_combined.shape[1]
    feature_pipeline.non_constant_mask = np.std(X_train, axis=0) != 0
    X_train_filtered = X_train[:, feature_pipeline.non_constant_mask]
    X_test_filtered = X_test[:, feature_pipeline.non_constant_mask]

    feature_pipeline.scaler = RobustScaler()
    X_train_scaled = feature_pipeline.scaler.fit_transform(X_train_filtered)
    X_test_scaled = feature_pipeline.scaler.transform(X_test_filtered)

    # Update feature groups after filtering
    new_feature_groups = {}
    start_idx = 0
    for group, count in feature_groups.items():
        end_idx = start_idx + count
        group_mask = feature_pipeline.non_constant_mask[start_idx:end_idx]
        new_feature_groups[group] = np.sum(group_mask)
        start_idx = end_idx

    print("Feature group sizes after removing constant features:")
    for group, size in new_feature_groups.items():
        print(f"  {group}: {size}")

    # Apply mRMR feature selection
    print("\nApplying mRMR feature selection...")
    selected_features, feature_distribution, importance_threshold = objective_group_importance_selection(
        X_train_scaled, y_train, new_feature_groups
    )

    X_train_selected = X_train_scaled[:, selected_features]
    X_test_selected = X_test_scaled[:, selected_features]

    feature_pipeline.selected_indices_ = selected_features
    feature_pipeline.mordred_columns = mordred_columns

    print(f"Selected {len(selected_features)} features")

    # Train neural network
    print("\nTraining neural network model...")
    nn_model = ImprovedGSK3NeuralNetwork(
        input_dim=X_train_selected.shape[1],
        hidden_dims=[256, 128, 64, 32],
        dropout_rates=[0.4, 0.4, 0.3, 0.3]
    ).to(device)

    batch_size = 64
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train_selected).to(device),
        torch.FloatTensor(y_train.reshape(-1, 1)).to(device)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(nn_model.parameters(), lr=1e-4, weight_decay=5e-4)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=4e-4,
        steps_per_epoch=len(train_loader),
        epochs=150,
        pct_start=0.3,
        div_factor=25,
        final_div_factor=10000
    )

    early_stopping = {
        'patience': 20,
        'min_delta': 0.001,
        'counter': 0,
        'best_loss': float('inf'),
        'best_model': None
    }

    nn_history = {'train_loss': [], 'val_loss': [], 'train_r2': [], 'val_r2': [], 'lr': []}

    # Training loop
    for epoch in range(150):
        nn_model.train()
        epoch_loss = 0
        batch_count = 0

        for X_batch, y_batch in train_loader:
            outputs = nn_model(X_batch)
            loss = criterion(outputs, y_batch)

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(nn_model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            batch_count += 1

        avg_loss = epoch_loss / batch_count

        nn_model.eval()
        with torch.no_grad():
            train_outputs = nn_model(torch.FloatTensor(X_train_selected).to(device))
            train_preds = train_outputs.cpu().numpy().flatten()
            train_r2 = r2_score(y_train, train_preds)

            val_outputs = nn_model(torch.FloatTensor(X_test_selected).to(device))
            val_preds = val_outputs.cpu().numpy().flatten()
            val_loss = criterion(val_outputs, torch.FloatTensor(y_test.reshape(-1, 1)).to(device)).item()
            val_r2 = r2_score(y_test, val_preds)

        nn_history['train_loss'].append(avg_loss)
        nn_history['val_loss'].append(val_loss)
        nn_history['train_r2'].append(train_r2)
        nn_history['val_r2'].append(val_r2)
        nn_history['lr'].append(optimizer.param_groups[0]['lr'])

        if epoch % 10 == 0 or epoch == 149:
            print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f}, R2: {train_r2:.4f} | "
                  f"Val Loss: {val_loss:.4f}, R2: {val_r2:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f}")

        overfitting_gap = train_r2 - val_r2

        if val_loss < early_stopping['best_loss'] - early_stopping['min_delta'] and overfitting_gap < 0.25:
            early_stopping['best_loss'] = val_loss
            early_stopping['counter'] = 0
            early_stopping['best_model'] = nn_model.state_dict().copy()
        else:
            early_stopping['counter'] += 1
            if early_stopping['counter'] >= early_stopping['patience']:
                print(f"Early stopping at epoch {epoch+1}")
                break

    if early_stopping['best_model'] is not None:
        nn_model.load_state_dict(early_stopping['best_model'])

    clear_memory()

    # Train ensemble
    print("\nTraining ensemble models...")

    X_train_part1, X_train_part2, y_train_part1, y_train_part2 = train_test_split(
        X_train_selected, y_train, test_size=0.3, random_state=42
    )

    ensemble = AdaptiveWeightEnsemble(
        base_models=[
            RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_split=12, min_samples_leaf=6, random_state=42),
            xgb.XGBRegressor(n_estimators=200, learning_rate=0.02, max_depth=5, subsample=0.7, colsample_bytree=0.7,
                             reg_alpha=0.05, reg_lambda=1.0, random_state=42),
            SVR(kernel='rbf', C=3.0, gamma='scale', epsilon=0.15),
            KNeighborsRegressor(n_neighbors=9, weights='distance'),
            ElasticNet(alpha=0.08, l1_ratio=0.7, max_iter=10000)
        ],
        meta_model=GradientBoostingRegressor(n_estimators=80, learning_rate=0.04, max_depth=2, subsample=0.7, random_state=42)
    )

    ensemble.nn_model = nn_model
    ensemble.fit(X_train_part1, y_train_part1, X_train_part2, y_train_part2)

    # Final evaluation
    print("\nEvaluating final model performance...")
    ensemble_preds = ensemble.predict(X_test_selected)
    ensemble_r2 = r2_score(y_test, ensemble_preds)
    ensemble_rmse = np.sqrt(mean_squared_error(y_test, ensemble_preds))

    print(f"Final model performance - R2: {ensemble_r2:.4f}, RMSE: {ensemble_rmse:.4f}")

    # Save models
    print("\nSaving models and components...")
    model_dir = f"{output_dir}/models"

    feature_pipeline.save(f"{model_dir}/preprocessing_pipeline.pkl")

    torch.save(nn_model.state_dict(), f"{model_dir}/nn_model.pt")
    nn_architecture = {
        'input_dim': X_train_selected.shape[1],
        'hidden_dims': [256, 128, 64, 32],
        'dropout_rates': [0.4, 0.4, 0.3, 0.3]
    }
    joblib.dump(nn_architecture, f"{model_dir}/nn_architecture.json")

    for i, expert in enumerate(ensemble.base_models):
        joblib.dump(expert, f"{model_dir}/expert_{i}.pkl")
    joblib.dump(ensemble.meta_model, f"{model_dir}/meta_model.pkl")

    ensemble.is_fitted = True
    joblib.dump(ensemble, f"{model_dir}/ensemble_complete.pkl")

    joblib.dump({'nn_weight': ensemble.nn_weight}, f"{model_dir}/nn_weight.pkl")

    print(f"Models saved to {model_dir}")

    # Create visualizations
    print("\nCreating performance visualizations...")
    vis_dir = f"{output_dir}/visualizations"

    # Actual vs predicted plot
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, ensemble_preds, alpha=0.7)

    min_val = min(min(y_test), min(ensemble_preds))
    max_val = max(max(y_test), max(ensemble_preds))
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2)

    plt.title('GSK3-beta Activity: Actual vs Predicted (mRMR)', fontsize=16)
    plt.xlabel('Actual pChEMBL Value', fontsize=12)
    plt.ylabel('Predicted pChEMBL Value', fontsize=12)
    plt.grid(True, alpha=0.3)

    plt.text(min_val + 0.5, max_val - 0.5,
             f"R2 = {ensemble_r2:.4f}\nRMSE = {ensemble_rmse:.4f}",
             fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig(f"{vis_dir}/actual_vs_predicted.png", dpi=300)
    plt.close()

    # Save performance summary
    total_time = time.time() - start_time
    performance_summary = {
        'execution_time_seconds': total_time,
        'execution_time_minutes': total_time/60,
        'r2_score': ensemble_r2,
        'rmse': ensemble_rmse,
        'training_samples': len(X_train),
        'test_samples': len(X_test),
        'total_features': X_combined.shape[1],
        'selected_features': X_train_selected.shape[1],
        'feature_distribution': {k: v for k, v in feature_distribution.items()},
        'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
    }

    joblib.dump(performance_summary, f"{output_dir}/performance_summary.json")

    print(f"\nTotal pipeline execution time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
    print(f"Results saved to '{output_dir}' directory")

    return {
        'model': ensemble,
        'nn_model': nn_model,
        'performance': {'r2': ensemble_r2, 'rmse': ensemble_rmse},
        'feature_pipeline': feature_pipeline,
        'feature_distribution': feature_distribution,
        'visualizations_path': vis_dir
    }

In [ ]:
#------------------------------------------------------------------------------
# 11. Resume from checkpoint function
#------------------------------------------------------------------------------

def resume_from_checkpoint(checkpoint_dir='gsk3_results/checkpoints'):
    """
    Resume ablation study from checkpoint
    """
    checkpoint_manager = CheckpointManager(checkpoint_dir)

    if not checkpoint_manager.checkpoint_exists():
        print("No checkpoint found. Please run the main pipeline first.")
        return None

    checkpoint = checkpoint_manager.load_checkpoint()

    print("Checkpoint information:")
    print(f"  Timestamp: {checkpoint['timestamp']}")
    print(f"  Completed experiments: {len(checkpoint['completed_experiments'])}/{checkpoint['total_experiments']}")
    print(f"  Elapsed time: {checkpoint['elapsed_time']:.2f} seconds ({checkpoint['elapsed_time']/3600:.2f} hours)")
    print(f"  Current experiment: {checkpoint['current_experiment']}")

    # Ask user to confirm resuming
    response = input("\nDo you want to resume from this checkpoint? (yes/no): ")

    if response.lower() != 'yes':
        print("Checkpoint resume cancelled.")
        return None

    print("\nResuming ablation study from checkpoint...")

    # Reconstruct data (you need to provide the data path)
    csv_file = 'GSK3B_in_vitro.csv'  # Update this path as needed
    output_dir = 'gsk3_results'

    # Load and preprocess data with encoding handling
    print(f"Loading data from {csv_file}...")

    try:
        df = load_csv_with_encoding(csv_file)
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        raise

    df = df.dropna(subset=['pChEMBL Value'])

    # Regenerate features (this part needs to match the original pipeline)
    print("Regenerating features...")
    mols_2d = [Chem.MolFromSmiles(smi) for smi in df['Smiles']]
    mols_3d = [generate_3d_conformation(mol) for mol in mols_2d]

    fps = [generate_ecfp_fingerprint(mol, radius=3, nBits=1024) for mol in mols_2d]
    fps = np.array(fps)

    descriptors_2d = [get_essential_2d_descriptors(mol) for mol in mols_2d]
    descriptors_2d = np.array(descriptors_2d)

    gsk3_desc = [enhanced_gsk3_descriptors(mol) for mol in mols_2d]
    gsk3_desc = np.array(gsk3_desc)

    descriptors_3d = [get_optimized_3d_descriptors(mol) for mol in mols_3d]
    descriptors_3d = np.array(descriptors_3d)

    calc = generate_optimized_mordred_descriptors()
    mordred_df = calc.pandas(mols_2d, quiet=True).fillna(0)
    numeric_cols = mordred_df.select_dtypes(include=[np.number]).columns
    mordred_desc = mordred_df[numeric_cols].values

    X_combined = np.hstack([
        fps,
        descriptors_2d,
        gsk3_desc,
        descriptors_3d,
        mordred_desc
    ])

    y = df['pChEMBL Value'].values

    # Resume ablation study
    run_ablation_study_with_checkpoints(
        X_combined, y, checkpoint['feature_groups'],
        output_dir=output_dir, n_cv_folds=5, checkpoint_interval=3600
    )

In [ ]:
#------------------------------------------------------------------------------
# 12. Main execution
#------------------------------------------------------------------------------

def test_csv_loading(csv_file='GSK3B_in_vitro.csv'):
    """
    Test CSV file loading and display basic information
    """
    print(f"Testing CSV file: {csv_file}")
    print("-" * 50)

    try:
        df = load_csv_with_encoding(csv_file)

        print(f"CSV loaded successfully!")
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"\nFirst 5 rows:")
        print(df.head())

        # Check for required columns
        required_columns = ['Smiles', 'pChEMBL Value']
        for col in required_columns:
            if col in df.columns:
                print(f"\n'{col}' column found - OK")
                print(f"  Non-null values: {df[col].notna().sum()}/{len(df)}")
            else:
                print(f"\n'{col}' column NOT FOUND - ERROR")

        return df

    except Exception as e:
        print(f"Error loading CSV: {e}")
        return None

In [ ]:
if __name__ == "__main__":
    print("GSK3-Beta QSAR Modeling with mRMR Feature Selection and Ablation Study")
    print("="*70)

    # Configuration
    csv_file = 'GSK3B_in_vitro.csv'
    output_dir = 'gsk3_results'
    run_ablation = True  # Set to False to skip ablation study

    # Test CSV loading first
    print("\nTesting CSV file loading...")
    test_df = test_csv_loading(csv_file)

    if test_df is None:
        print("\nCSV loading failed. Please check your file and try again.")
        exit(1)

    # Check if user wants to resume from checkpoint
    if os.path.exists(f"{output_dir}/checkpoints/ablation_checkpoint.pkl"):
        response = input("\nCheckpoint detected. Do you want to resume from checkpoint? (yes/no): ")
        if response.lower() == 'yes':
            resume_from_checkpoint(f"{output_dir}/checkpoints")
        else:
            # Run new pipeline
            results = run_mrmr_pipeline_with_ablation(csv_file, output_dir, run_ablation)

            if results:
                print("\nPipeline completed successfully!")
                print(f"Final R2 Score: {results['performance']['r2']:.4f}")
                print(f"Final RMSE: {results['performance']['rmse']:.4f}")
    else:
        # Run new pipeline
        results = run_mrmr_pipeline_with_ablation(csv_file, output_dir, run_ablation)

        if results:
            print("\nPipeline completed successfully!")
            print(f"Final R2 Score: {results['performance']['r2']:.4f}")
            print(f"Final RMSE: {results['performance']['rmse']:.4f}")

    print("\nAll tasks completed!")

GSK3-Beta QSAR Modeling with mRMR Feature Selection and Ablation Study

Testing CSV file loading...
Testing CSV file: GSK3B_in_vitro.csv
--------------------------------------------------
Successfully loaded CSV with latin-1 encoding
CSV loaded successfully!
Shape: (3532, 5)
Columns: ['Smiles', 'Standard Relation', 'pChEMBL Value', 'Assay Description', 'assay_type']

First 5 rows:
                                              Smiles Standard Relation  \
0                  CN(C)c1ccc(-c2cc3[nH]ccnc-3n2)cc1               '='   
1         NC1(Cc2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1               '='   
2          NC1(c2ccc(Cl)cc2)CCN(c2ncnc3[nH]cnc23)CC1               NaN   
3  Cc1ccc(N2CCN(CCCNC(=O)c3nc(-c4cccnc4)no3)CC2)cc1C               '='   
4  O=C(NCCCN1CCN(c2cccc(Cl)c2)CC1)c1nc(-c2ccncc2)no1               '='   

   pChEMBL Value                                  Assay Description assay_type  
0           4.92  Inhibition of glycogen synthase kinase-3 beta,...   in_vitro  
1        